# Duka Electronics: Half-Year Sales Review (Jan–Jun 2026)
Prepared for the Head of Sales. Source: `duka_sales.csv` (1,000 orders, seed 2026).

## Executive Summary
*(to be written after all tasks are complete)*

In [1]:
!git clone https://github.com/mangacindy9-code/plp-da-a3-duka-sales-and-agent-cleanup.git
%cd plp-da-a3-duka-sales-and-agent-cleanup

# Fix the invalid non-breaking space (U+00A0) in generate_data.py
with open('generate_data.py', 'r', encoding='utf-8') as f:
    content = f.read()

fixed_content = content.replace('\xa0', ' ')

with open('generate_data.py', 'w', encoding='utf-8') as f:
    f.write(fixed_content)

!python generate_data.py
!ls

fatal: destination path 'plp-da-a3-duka-sales-and-agent-cleanup' already exists and is not an empty directory.
/content/plp-da-a3-duka-sales-and-agent-cleanup
agent_cleanup.ipynb	generate_data.py
agents.csv		outputs
agent_transactions.csv	plp-da-a3-duka-sales-and-agent-cleanup
cleaning_pipeline.py	README.md
duka_sales.csv		requirements.txt


## 1. First-five-minutes inspection

In [2]:
import os
import pandas as pd, numpy as np

os.makedirs("outputs", exist_ok=True)
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", sales.shape)
display(sales.head())
print(sales.dtypes, "\n")
print("Missing values:\n", sales.isna().sum(), "\n")
print("Date range:", sales["order_date"].min().date(), "to", sales["order_date"].max().date())
print("Duplicate rows:", sales.duplicated().sum())
for c in ["branch", "product", "channel"]:
    print(f"\n{c}:\n", sales[c].value_counts())
display(sales[["quantity", "unit_price"]].describe())

Shape: (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object 

Missing values:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64 

Date range: 2026-01-01 to 2026-06-29
Duplicate rows: 0

branch:
 branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

product:
 product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64

channel:
 channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64


,quantity,unit_price
count,1000.000000,1000.000000
mean,3.012000,23918.300000
std,1.416991,23908.790546
min,1.000000,1200.000000
25%,2.000000,2500.000000
50%,3.000000,15000.000000
75%,4.000000,28000.000000
max,5.000000,65000.000000


- **Shape:** 1,000 rows × [N] columns.
- **Types:** `order_date` parsed as datetime64; [list the others].
- **Missing values:** [none / describe].
- **Date range:** [start] to [end].
- **Key categories:** 5 branches ([names]), [N] products ([names]), 3 channels (In-store, Online, Phone order).
- **Anything odd:** [duplicates, unusual quantities or prices, or "nothing unusual found"].

## 2. Derived columns

In [3]:
sales["revenue"]    = sales["quantity"] * sales["unit_price"]
sales["month"]      = sales["order_date"].dt.to_period("M")
sales["weekday"]    = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5      # Sat=5, Sun=6
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

display(sales[["order_date", "revenue", "month", "weekday", "is_weekend", "price_band"]].head())
print(sales["price_band"].value_counts())

,order_date,revenue,month,weekday,is_weekend,price_band
0,2026-06-03,45000,2026-06,Wednesday,False,Value
1,2026-02-02,260000,2026-02,Monday,False,Premium
2,2026-01-05,140000,2026-01,Monday,False,Premium
3,2026-04-26,12500,2026-04,Sunday,True,Value
4,2026-03-07,7500,2026-03,Saturday,True,Value


price_band
Value      565
Premium    435
Name: count, dtype: int64


Price band is based on **unit price** (≥ KES 28,000 = Premium). Revenue = quantity × unit price.

## 3. Branch ranking: revenue vs number of orders

In [4]:
branch = sales.groupby("branch").agg(total_revenue=("revenue", "sum"),
                                     orders=("revenue", "count"))
branch["revenue_rank"] = branch["total_revenue"].rank(ascending=False, method="min").astype(int)
branch["orders_rank"]  = branch["orders"].rank(ascending=False, method="min").astype(int)
branch["avg_order_value"] = (branch["total_revenue"] / branch["orders"]).round(0)
branch["revenue_share_%"] = (branch["total_revenue"] / branch["total_revenue"].sum() * 100).round(1)
branch.sort_values("total_revenue", ascending=False)

,total_revenue,orders,revenue_rank,orders_rank,avg_order_value,revenue_share_%
branch,,,,,,
Nairobi,26106100,376,1,1,69431.0,36.4
Mombasa,14003700,186,2,2,75289.0,19.5
Nakuru,11970400,162,3,3,73891.0,16.7
Kisumu,10868100,159,4,4,68353.0,15.1
Eldoret,8844200,117,5,5,75591.0,12.3


**Answer:** The rankings [are / are not] the same. [Branch] is first by revenue with [x]% of the total, and [Branch] is first by orders.
Any difference comes from average order value: [Branch] earns KES [..] per order versus KES [..] at [Branch], because [it sells more laptops / higher-priced items].
Revenue depends on how many orders there are *and* how large each one is.

## 4. Top three products: revenue vs units sold

In [5]:
prod = sales.groupby("product").agg(revenue=("revenue", "sum"),
                                    units=("quantity", "sum"),
                                    avg_unit_price=("unit_price", "mean")).round(0)
print("Top 3 by revenue:")
display(prod.nlargest(3, "revenue"))
print("Top 3 by units sold:")
display(prod.nlargest(3, "units"))
prod.sort_values("revenue", ascending=False)

Top 3 by revenue:


,revenue,units,avg_unit_price
product,,,
Laptop,41535000,639,65000.0
Tablet,19348000,691,28000.0
Phone,8880000,592,15000.0


Top 3 by units sold:


,revenue,units,avg_unit_price
product,,,
Tablet,19348000,691,28000.0
Laptop,41535000,639,65000.0
Phone,8880000,592,15000.0


,revenue,units,avg_unit_price
product,,,
Laptop,41535000,639,65000.0
Tablet,19348000,691,28000.0
Phone,8880000,592,15000.0
Headphones,1387500,555,2500.0
Charger,642000,535,1200.0


**Answer:** Top 3 by revenue: [..]. Top 3 by units: [..]. [Explain the difference: cheap items such as accessories sell many units but earn little,
while laptops sell fewer units at a high price and dominate revenue (KES 41.5M). If the two lists are identical, say so and explain why.]

## 5. Weekend vs weekday orders

In [6]:
wk = sales.groupby("is_weekend").agg(orders=("revenue", "count"),
                                     total_revenue=("revenue", "sum"),
                                     avg_order_revenue=("revenue", "mean")).round(0)
wk.index = wk.index.map({False: "Weekday", True: "Weekend"})

# Fair comparison: weekends are only 2 of every 7 days
days = sales.groupby("is_weekend")["order_date"].nunique()
days.index = days.index.map({False: "Weekday", True: "Weekend"})
wk["days_with_orders"] = days
wk["orders_per_day"] = (wk["orders"] / wk["days_with_orders"]).round(1)
wk

,orders,total_revenue,avg_order_revenue,days_with_orders,orders_per_day
is_weekend,,,,,
Weekday,723,51856200,71724.0,128,5.6
Weekend,277,19936300,71972.0,51,5.4


**Answer:** Weekdays had [N] orders and KES [..] revenue; weekends had [N] orders and KES [..]. Raw totals favour weekdays because there are
5 weekdays for every 2 weekend days, but per day weekends average [x] orders versus [y] on weekdays. Average order revenue is KES [..] on weekends versus KES [..] on weekdays.

## 6. Best and worst month

In [7]:
monthly = sales.groupby("month")["revenue"].sum()
display(monthly)

best, worst = monthly.idxmax(), monthly.idxmin()
pct_diff = (monthly[best] - monthly[worst]) / monthly[worst] * 100
print(f"Best: {best} (KES {monthly[best]:,.0f}) | Worst: {worst} (KES {monthly[worst]:,.0f})")
print(f"Best month is {pct_diff:.1f}% higher than the worst month")

# Check each month is complete, so the comparison is fair
display(sales.groupby("month")["order_date"].agg(["min", "max", "count"]))

,revenue
month,
2026-01,11786600
2026-02,11772000
2026-03,12396800
2026-04,13518300
2026-05,11415400
2026-06,10903400


Best: 2026-04 (KES 13,518,300) | Worst: 2026-06 (KES 10,903,400)
Best month is 24.0% higher than the worst month


,min,max,count
month,,,
2026-01,2026-01-01,2026-01-31,173
2026-02,2026-02-01,2026-02-28,159
2026-03,2026-03-01,2026-03-31,186
2026-04,2026-04-01,2026-04-30,167
2026-05,2026-05-01,2026-05-31,153
2026-06,2026-06-01,2026-06-29,162


**Answer:** [Month] was the best month (KES [..]) and [Month] the worst (KES [..]), a difference of [x]%, calculated as (best − worst) ÷ worst.
[Mention whether any month is incomplete, using the min/max dates table.]

## 7. Premium online orders from Nairobi and Mombasa

In [8]:
sub = sales[(sales["price_band"] == "Premium")
            & (sales["channel"] == "Online")
            & (sales["branch"].isin(["Nairobi", "Mombasa"]))]

print("Orders:", len(sub), "| Total revenue: KES", f"{sub['revenue'].sum():,.0f}",
      f"| {sub['revenue'].sum() / sales['revenue'].sum() * 100:.1f}% of all revenue")
display(sub.groupby("branch")["revenue"].agg(["count", "sum"]))

sub.to_csv("outputs/premium_online_coast_capital.csv", index=False)

chk = pd.read_csv("outputs/premium_online_coast_capital.csv")
assert set(chk["price_band"]) == {"Premium"} and set(chk["channel"]) == {"Online"}
assert set(chk["branch"]) <= {"Nairobi", "Mombasa"}
print("Saved:", chk.shape)

Orders: 83 | Total revenue: KES 10,218,000 | 14.2% of all revenue


,count,sum
branch,,
Mombasa,33,3696000
Nairobi,50,6522000


Saved: (83, 12)


**Answer:** There are [N] such orders worth KES [..], which is [x]% of total revenue. [Nairobi / Mombasa] contributes more.
The file is saved at `outputs/premium_online_coast_capital.csv`.